# 34 — Non-IID data: client drift

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/034-non-iid-and-convergence.ipynb)

When two clients disagree about the answer, local training pulls their models in opposite directions before the server averages them. This notebook measures that gap round by round, and shows why doing more local work makes the disagreement worse.

## 1. The numbers: two clients, opposite targets

Client 1 only sees zeros and client 2 only sees ones, so their ideal models are 0 and 1. Both start from the same server model, take one local step, and are then averaged. The distance between the two copies is what we will call drift.

In [ ]:
import numpy as np                                # the array library, preinstalled on Colab

targets = np.array([0.0, 1.0])                    # what each client would choose alone
sizes = np.array([100, 100])                      # both clients hold the same amount of data
shares = sizes / sizes.sum()                      # so each count weighs one half
w_start = 0.2                                     # the model the server sends out
lr = 0.5                                          # size of one local gradient step

local = w_start - lr * (w_start - targets)        # one local step for each client
w_next = float(shares @ local)                    # server: average the two copies
gap = float(local[1] - local[0])                  # drift = how far apart they ended up

print("copies after one local step:", local)      # [0.1 0.6]
print("global after the average:", round(w_next, 3))   # 0.35
print("drift (gap between clients):", round(gap, 3))   # 0.5

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy, which Colab already has.</li><li><code>targets = ...</code> — client 1 wants 0, client 2 wants 1.</li><li><code>sizes = ...</code> — the two clients hold equally many examples.</li><li><code>shares = ...</code> — each client's fraction of the data, here 0.5.</li><li><code>w_start = 0.2</code> — the shared starting model.</li><li><code>lr = 0.5</code> — how far one local step moves the model.</li><li><code>local = ...</code> — apply one gradient step to both copies at once.</li><li><code>w_next = ...</code> — the server's average of the two copies.</li><li><code>gap = ...</code> — the subtraction gives the drift after this round.</li><li><code>print(...)</code> — shows the two copies before averaging.</li><li><code>print(...)</code> — the global value the server produced.</li><li><code>print(...)</code> — the size of the gap, the number to watch.</li></ul>

</details>

## 2. Step by step: eight rounds of local-then-average

Each round repeats the same two moves: every client trains from the broadcast model, then the server averages. Record the pre-average copy of each client, because that is where the disagreement lives.

In [ ]:
w = 0.2                                          # the server's starting model
local = np.array([0.2, 0.2])                      # both clients begin from the same copy
c1, c2, globals_ = [], [], []                     # history: client 1, client 2, server

for r in range(8):                                # eight communication rounds
    local = local - lr * (local - targets)        # local training happens here
    c1.append(float(local[0]))                    # client 1's copy before averaging
    c2.append(float(local[1]))                    # client 2's copy before averaging
    w = float(shares @ local)                     # server averages the copies
    globals_.append(w)                            # record the global model
    local = np.array([w, w])                      # broadcast it back to both clients
    print(f"round {r+1}: c1={c1[-1]:.3f} c2={c2[-1]:.3f} global={w:.3f}")   # readout

print("final drift:", round(c2[-1] - c1[-1], 3))   # 0.5, unchanged

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>w = 0.2</code> — the server starts where section 1 started.</li><li><code>local = ...</code> — two copies of that model, one per client.</li><li><code>c1, c2, globals_ = ...</code> — three empty lists for the three curves.</li><li><code>for r in range(8)</code> — eight rounds of training and averaging.</li><li><code>local = ...</code> — the local gradient step, applied to both clients.</li><li><code>c1.append(...)</code> — save client 1's copy before it is overwritten.</li><li><code>c2.append(...)</code> — save client 2's copy the same way.</li><li><code>w = ...</code> — the server's average, weighted by data size.</li><li><code>globals_.append(w)</code> — store the global value for the plot.</li><li><code>local = ...</code> — both clients restart from the averaged model.</li><li><code>print(f"round ...")</code> — the two copies and the global value each round.</li><li><code>print("final drift"...)</code> — the gap at the end of training.</li></ul>

</details>

## 3. Plot: the clients pull apart

The two coloured curves are what each client's model looks like just before the average. The dashed line is the server's model; the shaded band is the drift. The band does not shrink, because the local step always undoes the average.

In [ ]:
import matplotlib.pyplot as plt                   # plotting library, preinstalled on Colab

rounds = np.arange(1, 9)                          # x-axis: the eight rounds
plt.figure(figsize=(6, 3))                        # a wide, short figure
plt.plot(rounds, c1, "o-", color="tab:blue", label="client 1 (target 0)")    # low copy
plt.plot(rounds, c2, "s-", color="tab:orange", label="client 2 (target 1)")  # high copy
plt.plot(rounds, globals_, "k--", label="global w")                          # server
plt.fill_between(rounds, c1, c2, color="grey", alpha=0.15, label="drift")    # the gap
plt.ylim(-0.05, 1.05)                             # fixed range, so nothing jumps
plt.xlabel("communication round")                 # x-axis label
plt.ylabel("model w")                             # y-axis label
plt.legend(loc="center right")                    # name all four artists
plt.title("clients pull apart; the average sits in the middle")   # the takeaway
plt.show()                                        # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — loads the plotting library.</li><li><code>rounds = ...</code> — one x value per recorded round.</li><li><code>plt.figure(...)</code> — a fresh canvas.</li><li><code>plt.plot(...)</code> — client 1's copy per round.</li><li><code>plt.plot(...)</code> — client 2's copy per round.</li><li><code>plt.plot(...)</code> — the global model the server sends out.</li><li><code>plt.fill_between(...)</code> — shade the space between the two clients.</li><li><code>plt.ylim(...)</code> — keep the full 0-to-1 range visible.</li><li><code>plt.xlabel(...)</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.legend(...)</code> — a key for the three lines and the band.</li><li><code>plt.title(...)</code> — the takeaway.</li><li><code>plt.show()</code> — draws the figure.</li></ul>

</details>

## 4. Interactive: local epochs on a slider

The slider sets how many local steps each client takes before the server averages. One epoch keeps the band at 0.5; eight epochs opens it almost to the full range. The title prints the final drift.

In [ ]:
from ipywidgets import interact                      # sliders that rerun a function

def simulate(epochs=1, rounds=8):                   # one run of local-then-average
    w = 0.2                                         # server's starting model
    local = np.array([0.2, 0.2])                    # two client copies
    g_hist, a_hist, b_hist = [], [], []             # global, client 1, client 2
    for r in range(rounds):                         # the outer loop over rounds
        for e in range(epochs):                     # the inner loop over local steps
            local = local - lr * (local - targets)  # one local gradient step
        a_hist.append(float(local[0]))              # copy 1 before averaging
        b_hist.append(float(local[1]))              # copy 2 before averaging
        w = float(shares @ local)                   # weighted server average
        g_hist.append(w)                            # store the global value
        local = np.array([w, w])                    # broadcast back to the clients
    return g_hist, a_hist, b_hist                   # hand the histories to the caller

def show(epochs=1):                                 # redraws for one slider value
    g_hist, a_hist, b_hist = simulate(epochs)       # run the simulation
    plt.figure(figsize=(6, 3))                      # fresh figure per drag
    plt.plot(range(1, 9), a_hist, "o-", color="tab:blue", label="client 1")     # low copy
    plt.plot(range(1, 9), b_hist, "s-", color="tab:orange", label="client 2")   # high copy
    plt.plot(range(1, 9), g_hist, "k--", label="global w")                      # server
    plt.fill_between(range(1, 9), a_hist, b_hist, color="grey", alpha=0.15)     # drift band
    plt.ylim(-0.05, 1.05)                           # fixed axes, so nothing jumps
    plt.xlabel("communication round")               # x-axis label
    plt.ylabel("model w")                           # y-axis label
    plt.legend(loc="center right")                  # name the lines
    plt.title(f"{epochs} epoch(s): final drift = {b_hist[-1] - a_hist[-1]:.2f}")   # readout
    plt.show()                                      # draw it

interact(show, epochs=(1, 8, 1))                    # slider: 1 to 8 local epochs

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — Colab ships the widgets library.</li><li><code>def simulate(epochs=1, rounds=8)</code> — one full run, returning three histories.</li><li><code>w = 0.2</code> — every run starts from the same server model.</li><li><code>local = ...</code> — the two client copies.</li><li><code>g_hist, a_hist, b_hist = ...</code> — empty lists for global and the two clients.</li><li><code>for r in range(rounds)</code> — the communication rounds.</li><li><code>for e in range(epochs)</code> — the local steps inside each round.</li><li><code>local = ...</code> — one gradient step for both clients.</li><li><code>a_hist.append(...)</code> — client 1's copy before averaging.</li><li><code>b_hist.append(...)</code> — client 2's copy before averaging.</li><li><code>w = ...</code> — the server's weighted average.</li><li><code>g_hist.append(w)</code> — record the global value.</li><li><code>local = ...</code> — broadcast the average to both clients.</li><li><code>return ...</code> — the three lists the plotting code needs.</li><li><code>def show(epochs=1)</code> — the function attached to the slider.</li><li><code>g_hist, ... = simulate(epochs)</code> — run with the chosen number of epochs.</li><li><code>plt.figure(...)</code> — a new canvas for each slider position.</li><li><code>plt.plot(...)</code> — client 1's curve.</li><li><code>plt.plot(...)</code> — client 2's curve.</li><li><code>plt.plot(...)</code> — the global model.</li><li><code>plt.fill_between(...)</code> — shade the drift.</li><li><code>plt.ylim(-0.05, 1.05)</code> — fixed axes, so nothing jumps.</li><li><code>plt.xlabel("communication round")</code> — names the horizontal axis.</li><li><code>plt.ylabel("model w")</code> — names the vertical axis.</li><li><code>plt.legend(...)</code> — a key for the three lines.</li><li><code>plt.title(...)</code> — live readout of epochs and final drift.</li><li><code>plt.show()</code> — draw the frame.</li><li><code>interact(show, epochs=(1, 8, 1))</code> — the slider, its range and step.</li></ul>

</details>

## 5. When it breaks: more local epochs, more drift

More local work makes each client's own model better and the shared model worse. At eight epochs each client sits on its target while the average is 0.5, the worst possible value for both. The curve below is that gap as a function of local epochs.

In [ ]:
import numpy as np                                # reload the array library
import matplotlib.pyplot as plt                   # reload the plotting library

epochs = np.arange(1, 13)                         # try 1 to 12 local epochs
drift = []                                        # one gap per setting
for e in epochs:                                  # one simulation per setting
    _, a_hist, b_hist = simulate(int(e))          # ignore the global history here
    drift.append(b_hist[-1] - a_hist[-1])         # gap at the end of round eight

print("global w is 0.5, loss on client 1:", float((0.5 - targets[0]) ** 2))   # 0.25
print("global w is 0.5, loss on client 2:", float((0.5 - targets[1]) ** 2))   # 0.25
print("a client's own model has loss 0, but it is never shared")   # perfect, and discarded
print("drift at 1 epoch:", round(drift[0], 3), "at 12 epochs:", round(drift[-1], 3))   # 0.5 -> 1.0

plt.figure(figsize=(6, 3))                        # a wide, short figure
plt.plot(epochs, drift, "o-", color="tab:red")    # drift against local epochs
plt.axhline(1.0, color="grey", linestyle="--")    # the largest possible gap
plt.ylim(0, 1.05)                                 # fixed range
plt.xlabel("local epochs per round")              # x-axis label
plt.ylabel("distance between the two client models")   # y-axis label
plt.title("more local epochs -> clients drift to opposite ends")   # the takeaway
plt.show()                                        # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — the array library, reloaded for this cell.</li><li><code>import matplotlib.pyplot as plt</code> — the plotting library.</li><li><code>epochs = ...</code> — the values we will try, as an array.</li><li><code>drift = []</code> — collects one gap per value.</li><li><code>for e in epochs</code> — one full simulation per number of epochs.</li><li><code>_, a_hist, b_hist = simulate(int(e))</code> — run it; the underscore drops the global list.</li><li><code>drift.append(...)</code> — the gap at the end of the last round.</li><li><code>print(...)</code> — the loss the shared model pays on client 1.</li><li><code>print(...)</code> — the same loss on client 2.</li><li><code>print(...)</code> — notes that each local model is perfect but discarded.</li><li><code>print(...)</code> — the gap at the two ends of the range.</li><li><code>plt.figure(...)</code> — a fresh canvas.</li><li><code>plt.plot(...)</code> — drift as a function of local epochs.</li><li><code>plt.axhline(1.0, ...)</code> — the ceiling: the clients' targets.</li><li><code>plt.ylim(...)</code> — keeps the ceiling in view.</li><li><code>plt.xlabel(...)</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.title(...)</code> — the takeaway.</li><li><code>plt.show()</code> — draws the figure.</li></ul>

</details>

## Your turn

Set `lr = 1.0` in section 1 and predict the two copies before rerunning — what happens to the average? Then drag section 4 to eight epochs and explain why the global line barely moves while the shaded band widens.